In [37]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("../data/raw")

train_df = pd.read_csv(
    data_dir / "UNSW_NB15_training-set.csv"
)

print("Training shape:", train_df.shape)
print("Number of classes:", train_df["attack_cat"].nunique())
print("Missing values:", train_df.isnull().sum().sum())

Training shape: (175341, 45)
Number of classes: 10
Missing values: 0


In [38]:
feature_columns = train_df.columns.drop(["id", "attack_cat", "label"])

feature_hash = pd.util.hash_pandas_object(
    train_df[feature_columns],
    index=False
)

conflicting_groups = (
    train_df.groupby(feature_hash)["attack_cat"]
    .nunique()
    .gt(1)
    .sum()
)

print("Unique feature patterns:", feature_hash.nunique())
print("Repeated feature rows:", feature_hash.duplicated().sum())
print("Conflicting label groups:", conflicting_groups)

Unique feature patterns: 101040
Repeated feature rows: 74301
Conflicting label groups: 1772


In [39]:
conflict_counts = train_df.groupby(feature_hash)["attack_cat"].nunique()

conflict_mask = feature_hash.map(conflict_counts.gt(1))

print("Rows in conflicting groups:", int(conflict_mask.sum()))
print("Conflicting row percentage:", f"{conflict_mask.mean():.2%}")
print("Conflicting rows by class:")
print(train_df.loc[conflict_mask, "attack_cat"].value_counts())

Rows in conflicting groups: 30528
Conflicting row percentage: 17.41%
Conflicting rows by class:
attack_cat
Exploits          13299
DoS                9746
Fuzzers            2078
Reconnaissance     1889
Analysis           1441
Backdoor           1340
Normal              396
Generic             319
Worms                13
Shellcode             7
Name: count, dtype: int64


In [40]:
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df[feature_columns],
        train_df["attack_cat"],
        groups=feature_hash
    )
)

dev_train_df = train_df.iloc[train_idx].copy()
val_df = train_df.iloc[val_idx].copy()

group_overlap = len(
    set(feature_hash.iloc[train_idx]) &
    set(feature_hash.iloc[val_idx])
)

print("Training rows:", len(dev_train_df))
print("Validation rows:", len(val_df))
print("Feature-group overlap:", group_overlap)
print("Validation class distribution:")
print(val_df["attack_cat"].value_counts())

Training rows: 140273
Validation rows: 35068
Feature-group overlap: 0
Validation class distribution:
attack_cat
Normal            11200
Generic            8000
Exploits           6679
Fuzzers            3636
DoS                2453
Reconnaissance     2098
Analysis            400
Backdoor            349
Shellcode           227
Worms                26
Name: count, dtype: int64


In [41]:
X_dev = dev_train_df.drop(columns=["id", "attack_cat", "label"])
y_dev = dev_train_df["attack_cat"]

X_val = val_df.drop(columns=["id", "attack_cat", "label"])
y_val = val_df["attack_cat"]

print("X_dev:", X_dev.shape)
print("y_dev:", y_dev.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("Training class distribution:")
print(y_dev.value_counts())

X_dev: (140273, 42)
y_dev: (140273,)
X_val: (35068, 42)
y_val: (35068,)
Training class distribution:
attack_cat
Normal            44800
Generic           32000
Exploits          26714
Fuzzers           14548
DoS                9811
Reconnaissance     8393
Analysis           1600
Backdoor           1397
Shellcode           906
Worms               104
Name: count, dtype: int64


In [42]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

categorical_features = ["proto", "service", "state"]

numeric_features = X_dev.select_dtypes(include="number").columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [43]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

rf_control = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    ))
])

rf_control.fit(X_dev, y_dev)

print("Unweighted Random Forest trained successfully!")

Unweighted Random Forest trained successfully!


In [44]:
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score

y_val_pred_control = rf_control.predict(X_val)

print("Macro-F1:", f1_score(y_val, y_val_pred_control, average="macro"))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_pred_control))

print(classification_report(
    y_val,
    y_val_pred_control,
    digits=4,
    zero_division=0
))

Macro-F1: 0.5557389130422938
Balanced Accuracy: 0.538244226541566
                precision    recall  f1-score   support

      Analysis     0.3926    0.1325    0.1981       400
      Backdoor     0.2421    0.1318    0.1707       349
           DoS     0.4787    0.1146    0.1849      2453
      Exploits     0.6311    0.9177    0.7479      6679
       Fuzzers     0.7181    0.7349    0.7264      3636
       Generic     0.9983    0.9795    0.9888      8000
        Normal     0.9354    0.9142    0.9247     11200
Reconnaissance     0.9244    0.7517    0.8291      2098
     Shellcode     0.6204    0.5903    0.6050       227
         Worms     0.4286    0.1154    0.1818        26

      accuracy                         0.8261     35068
     macro avg     0.6370    0.5382    0.5557     35068
  weighted avg     0.8212    0.8261    0.8092     35068



In [45]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_weighted = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

rf_weighted.fit(X_dev, y_dev)

print("Class-weighted Random Forest trained successfully!")

Class-weighted Random Forest trained successfully!


In [46]:
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score

y_val_pred_weighted = rf_weighted.predict(X_val)

print("Macro-F1:", f1_score(y_val, y_val_pred_weighted, average="macro"))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_pred_weighted))

print(classification_report(
    y_val,
    y_val_pred_weighted,
    digits=4,
    zero_division=0
))

Macro-F1: 0.6034150134673405
Balanced Accuracy: 0.6655074342785456
                precision    recall  f1-score   support

      Analysis     0.1331    0.3150    0.1871       400
      Backdoor     0.0682    0.6017    0.1225       349
           DoS     0.3771    0.3620    0.3694      2453
      Exploits     0.8663    0.5296    0.6573      6679
       Fuzzers     0.6419    0.8559    0.7336      3636
       Generic     0.9989    0.9785    0.9886      8000
        Normal     0.9837    0.8472    0.9104     11200
Reconnaissance     0.8609    0.7965    0.8274      2098
     Shellcode     0.5644    0.7533    0.6453       227
         Worms     0.5714    0.6154    0.5926        26

      accuracy                         0.7713     35068
     macro avg     0.6066    0.6655    0.6034     35068
  weighted avg     0.8577    0.7713    0.8008     35068



In [47]:
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

weight_comparison = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Method": "None",
        "Accuracy": accuracy_score(y_val, y_val_pred_control),
        "Macro-F1": f1_score(y_val, y_val_pred_control, average="macro"),
        "Balanced Accuracy": balanced_accuracy_score(y_val, y_val_pred_control)
    },
    {
        "Model": "Random Forest",
        "Method": "Class Weight",
        "Accuracy": accuracy_score(y_val, y_val_pred_weighted),
        "Macro-F1": f1_score(y_val, y_val_pred_weighted, average="macro"),
        "Balanced Accuracy": balanced_accuracy_score(y_val, y_val_pred_weighted)
    }
])

print(weight_comparison.round(4).to_string(index=False))

        Model       Method  Accuracy  Macro-F1  Balanced Accuracy
Random Forest         None    0.8261    0.5557             0.5382
Random Forest Class Weight    0.7713    0.6034             0.6655


In [48]:
from pathlib import Path

output_dir = Path("../results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

weight_comparison.to_csv(
    output_dir / "class_weight_validation_comparison.csv",
    index=False
)

print("Class Weight comparison saved successfully!")

Class Weight comparison saved successfully!


In [49]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

rf_oversampled = ImbPipeline([
    ("preprocessor", clone(preprocessor)),
    ("oversampler", RandomOverSampler(random_state=42)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    ))
])

print("Random Oversampling pipeline created successfully!")

Random Oversampling pipeline created successfully!


In [50]:
rf_oversampled.fit(X_dev, y_dev)

print("Random Oversampling training completed successfully!")

Random Oversampling training completed successfully!


In [51]:
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score

y_val_pred_oversampled = rf_oversampled.predict(X_val)

print("Macro-F1:", f1_score(y_val, y_val_pred_oversampled, average="macro"))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_pred_oversampled))

print(classification_report(
    y_val,
    y_val_pred_oversampled,
    digits=4,
    zero_division=0
))

Macro-F1: 0.5949327724267537
Balanced Accuracy: 0.6286655477622186
                precision    recall  f1-score   support

      Analysis     0.0931    0.2450    0.1349       400
      Backdoor     0.0699    0.5530    0.1241       349
           DoS     0.3790    0.3657    0.3722      2453
      Exploits     0.8470    0.5537    0.6696      6679
       Fuzzers     0.6912    0.7830    0.7342      3636
       Generic     0.9987    0.9795    0.9890      8000
        Normal     0.9525    0.8924    0.9215     11200
Reconnaissance     0.8999    0.7712    0.8306      2098
     Shellcode     0.5984    0.6432    0.6200       227
         Worms     0.6190    0.5000    0.5532        26

      accuracy                         0.7797     35068
     macro avg     0.6149    0.6287    0.5949     35068
  weighted avg     0.8515    0.7797    0.8065     35068



In [52]:
oversampling_result = pd.DataFrame([{
    "Model": "Random Forest",
    "Method": "Random Oversampling",
    "Accuracy": accuracy_score(y_val, y_val_pred_oversampled),
    "Macro-F1": f1_score(y_val, y_val_pred_oversampled, average="macro"),
    "Balanced Accuracy": balanced_accuracy_score(y_val, y_val_pred_oversampled)
}])

imbalance_comparison = pd.concat(
    [weight_comparison, oversampling_result],
    ignore_index=True
)

print(imbalance_comparison.round(4).to_string(index=False))

        Model              Method  Accuracy  Macro-F1  Balanced Accuracy
Random Forest                None    0.8261    0.5557             0.5382
Random Forest        Class Weight    0.7713    0.6034             0.6655
Random Forest Random Oversampling    0.7797    0.5949             0.6287


In [53]:
from pathlib import Path

output_dir = Path("../results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

imbalance_comparison.to_csv(
    output_dir / "imbalance_validation_comparison.csv",
    index=False
)

print("Imbalance comparison saved successfully!")

Imbalance comparison saved successfully!


In [54]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

rf_undersampled = ImbPipeline([
    ("preprocessor", clone(preprocessor)),
    ("undersampler", RandomUnderSampler(
        sampling_strategy={
            "Normal": 15000,
            "Generic": 15000,
            "Exploits": 15000
        },
        random_state=42
    )),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    ))
])

print("Random Undersampling pipeline created successfully!")

Random Undersampling pipeline created successfully!


In [55]:
rf_undersampled.fit(X_dev, y_dev)

print("Random Undersampling training completed successfully!")

Random Undersampling training completed successfully!


In [56]:
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score

y_val_pred_undersampled = rf_undersampled.predict(X_val)

print("Macro-F1:", f1_score(y_val, y_val_pred_undersampled, average="macro"))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_pred_undersampled))

print(classification_report(
    y_val,
    y_val_pred_undersampled,
    digits=4,
    zero_division=0
))

Macro-F1: 0.6060085378604984
Balanced Accuracy: 0.61170336600114
                precision    recall  f1-score   support

      Analysis     0.3373    0.2100    0.2589       400
      Backdoor     0.2537    0.1461    0.1855       349
           DoS     0.3539    0.7583    0.4826      2453
      Exploits     0.8133    0.5799    0.6770      6679
       Fuzzers     0.6257    0.8718    0.7286      3636
       Generic     0.9994    0.9789    0.9890      8000
        Normal     0.9841    0.8390    0.9058     11200
Reconnaissance     0.8753    0.7898    0.8304      2098
     Shellcode     0.6322    0.6740    0.6525       227
         Worms     0.5000    0.2692    0.3500        26

      accuracy                         0.8008     35068
     macro avg     0.6375    0.6117    0.6060     35068
  weighted avg     0.8500    0.8008    0.8121     35068



In [57]:
undersampling_result = pd.DataFrame([{
    "Model": "Random Forest",
    "Method": "Random Undersampling",
    "Accuracy": accuracy_score(y_val, y_val_pred_undersampled),
    "Macro-F1": f1_score(y_val, y_val_pred_undersampled, average="macro"),
    "Balanced Accuracy": balanced_accuracy_score(y_val, y_val_pred_undersampled)
}])

imbalance_comparison = pd.concat(
    [imbalance_comparison, undersampling_result],
    ignore_index=True
)

print(imbalance_comparison.round(4).to_string(index=False))

        Model               Method  Accuracy  Macro-F1  Balanced Accuracy
Random Forest                 None    0.8261    0.5557             0.5382
Random Forest         Class Weight    0.7713    0.6034             0.6655
Random Forest  Random Oversampling    0.7797    0.5949             0.6287
Random Forest Random Undersampling    0.8008    0.6060             0.6117


In [58]:
from pathlib import Path

output_dir = Path("../results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

imbalance_comparison.to_csv(
    output_dir / "imbalance_validation_comparison.csv",
    index=False
)

print("Four-method comparison saved successfully!")

Four-method comparison saved successfully!


In [59]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

smote_numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

smote_categorical = SimpleImputer(
    strategy="most_frequent"
)

smote_preprocessor = ColumnTransformer([
    ("numeric", smote_numeric, numeric_features),
    ("categorical", smote_categorical, categorical_features)
])

print("SMOTENC preprocessing pipeline created successfully!")

SMOTENC preprocessing pipeline created successfully!


In [60]:
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_indices = list(
    range(len(numeric_features), len(numeric_features) + len(categorical_features))
)

sampling_strategy = {
    label: 5000
    for label, count in y_dev.value_counts().items()
    if count < 5000
}

post_encoder = ColumnTransformer([
    ("numeric", "passthrough", list(range(len(numeric_features)))),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_indices)
])

rf_smotenc = ImbPipeline([
    ("preprocessor", clone(smote_preprocessor)),
    ("sampler", SMOTENC(
        categorical_features=categorical_indices,
        sampling_strategy=sampling_strategy,
        random_state=42,
        k_neighbors=5
    )),
    ("encoder", post_encoder),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    ))
])

print("SMOTENC pipeline created successfully!")
print("Sampling strategy:", sampling_strategy)

SMOTENC pipeline created successfully!
Sampling strategy: {'Analysis': 5000, 'Backdoor': 5000, 'Shellcode': 5000, 'Worms': 5000}


In [61]:
rf_smotenc.fit(X_dev, y_dev)

print("SMOTENC training completed successfully!")

SMOTENC training completed successfully!


In [62]:
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score

y_val_pred_smotenc = rf_smotenc.predict(X_val)

print("Macro-F1:", f1_score(y_val, y_val_pred_smotenc, average="macro"))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_pred_smotenc))

print(classification_report(
    y_val,
    y_val_pred_smotenc,
    digits=4,
    zero_division=0
))

Macro-F1: 0.5638465664738073
Balanced Accuracy: 0.6020537288522831
                precision    recall  f1-score   support

      Analysis     0.1550    0.2375    0.1876       400
      Backdoor     0.0849    0.3352    0.1355       349
           DoS     0.5580    0.0844    0.1466      2453
      Exploits     0.6634    0.8203    0.7336      6679
       Fuzzers     0.7312    0.7338    0.7325      3636
       Generic     0.9986    0.9796    0.9890      8000
        Normal     0.9400    0.9115    0.9255     11200
Reconnaissance     0.9207    0.7526    0.8282      2098
     Shellcode     0.5455    0.8194    0.6549       227
         Worms     0.2727    0.3462    0.3051        26

      accuracy                         0.8095     35068
     macro avg     0.5870    0.6021    0.5638     35068
  weighted avg     0.8306    0.8095    0.8046     35068



In [63]:
smotenc_result = pd.DataFrame([{
    "Model": "Random Forest",
    "Method": "SMOTENC",
    "Accuracy": accuracy_score(y_val, y_val_pred_smotenc),
    "Macro-F1": f1_score(y_val, y_val_pred_smotenc, average="macro"),
    "Balanced Accuracy": balanced_accuracy_score(y_val, y_val_pred_smotenc)
}])

imbalance_comparison = pd.concat(
    [imbalance_comparison, smotenc_result],
    ignore_index=True
)

imbalance_comparison.to_csv(
    output_dir / "imbalance_validation_comparison.csv",
    index=False
)

print(imbalance_comparison.round(4).to_string(index=False))
print("Five-method comparison saved successfully!")

        Model               Method  Accuracy  Macro-F1  Balanced Accuracy
Random Forest                 None    0.8261    0.5557             0.5382
Random Forest         Class Weight    0.7713    0.6034             0.6655
Random Forest  Random Oversampling    0.7797    0.5949             0.6287
Random Forest Random Undersampling    0.8008    0.6060             0.6117
Random Forest              SMOTENC    0.8095    0.5638             0.6021
Five-method comparison saved successfully!


In [64]:
from sklearn.metrics import recall_score

classes = sorted(y_val.unique())

recall_comparison = pd.DataFrame({
    "Class": classes,
    "Baseline": recall_score(y_val, y_val_pred_control, labels=classes, average=None),
    "Class Weight": recall_score(y_val, y_val_pred_weighted, labels=classes, average=None),
    "Oversampling": recall_score(y_val, y_val_pred_oversampled, labels=classes, average=None),
    "Undersampling": recall_score(y_val, y_val_pred_undersampled, labels=classes, average=None),
    "SMOTENC": recall_score(y_val, y_val_pred_smotenc, labels=classes, average=None)
})

print(recall_comparison.round(4).to_string(index=False))

         Class  Baseline  Class Weight  Oversampling  Undersampling  SMOTENC
      Analysis    0.1325        0.3150        0.2450         0.2100   0.2375
      Backdoor    0.1318        0.6017        0.5530         0.1461   0.3352
           DoS    0.1146        0.3620        0.3657         0.7583   0.0844
      Exploits    0.9177        0.5296        0.5537         0.5799   0.8203
       Fuzzers    0.7349        0.8559        0.7830         0.8718   0.7338
       Generic    0.9795        0.9785        0.9795         0.9789   0.9796
        Normal    0.9142        0.8472        0.8924         0.8390   0.9115
Reconnaissance    0.7517        0.7965        0.7712         0.7898   0.7526
     Shellcode    0.5903        0.7533        0.6432         0.6740   0.8194
         Worms    0.1154        0.6154        0.5000         0.2692   0.3462


In [65]:
from pathlib import Path

output_dir = Path("../results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

recall_comparison.to_csv(
    output_dir / "class_recall_comparison.csv",
    index=False
)

print("Class recall comparison saved successfully!")

Class recall comparison saved successfully!


In [66]:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(
        train_df[feature_columns],
        train_df["attack_cat"],
        groups=feature_hash
    ),
    start=1
):
    val_counts = train_df.iloc[val_idx]["attack_cat"].value_counts()

    overlap = len(
        set(feature_hash.iloc[train_idx]) &
        set(feature_hash.iloc[val_idx])
    )

    fold_results.append({
        "Fold": fold,
        "Train": len(train_idx),
        "Validation": len(val_idx),
        "Worms": val_counts.get("Worms", 0),
        "Backdoor": val_counts.get("Backdoor", 0),
        "Group Overlap": overlap
    })

print(pd.DataFrame(fold_results).to_string(index=False))

 Fold  Train  Validation  Worms  Backdoor  Group Overlap
    1 140273       35068     26       349              0
    2 140272       35069     26       350              0
    3 140273       35068     26       349              0
    4 140273       35068     26       349              0
    5 140273       35068     26       349              0


In [67]:
from sklearn.model_selection import cross_validate
from sklearn.base import clone

cv_control_results = cross_validate(
    clone(rf_control),
    train_df[feature_columns],
    train_df["attack_cat"],
    groups=feature_hash,
    cv=cv,
    scoring={
        "macro_f1": "f1_macro",
        "balanced_accuracy": "balanced_accuracy"
    },
    n_jobs=1,
    error_score="raise"
)

cv_control_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Macro-F1": cv_control_results["test_macro_f1"],
    "Balanced Accuracy": cv_control_results["test_balanced_accuracy"]
})

print(cv_control_summary.round(4).to_string(index=False))
print("Mean Macro-F1:", cv_control_summary["Macro-F1"].mean())
print("Mean Balanced Accuracy:", cv_control_summary["Balanced Accuracy"].mean())
print("Std Macro-F1:", cv_control_summary["Macro-F1"].std())

 Fold  Macro-F1  Balanced Accuracy
    1    0.5557             0.5382
    2    0.5836             0.5532
    3    0.5576             0.5402
    4    0.5615             0.5400
    5    0.5496             0.5365
Mean Macro-F1: 0.5616098964881449
Mean Balanced Accuracy: 0.5416306232821341
Std Macro-F1: 0.013022765859046588


In [68]:
from sklearn.model_selection import cross_validate
from sklearn.base import clone

cv_weighted_results = cross_validate(
    clone(rf_weighted),
    train_df[feature_columns],
    train_df["attack_cat"],
    groups=feature_hash,
    cv=cv,
    scoring={
        "macro_f1": "f1_macro",
        "balanced_accuracy": "balanced_accuracy"
    },
    n_jobs=1,
    error_score="raise"
)

cv_weighted_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Macro-F1": cv_weighted_results["test_macro_f1"],
    "Balanced Accuracy": cv_weighted_results["test_balanced_accuracy"]
})

print(cv_weighted_summary.round(4).to_string(index=False))
print("Mean Macro-F1:", cv_weighted_summary["Macro-F1"].mean())
print("Mean Balanced Accuracy:", cv_weighted_summary["Balanced Accuracy"].mean())
print("Std Macro-F1:", cv_weighted_summary["Macro-F1"].std())

 Fold  Macro-F1  Balanced Accuracy
    1    0.6034             0.6655
    2    0.5937             0.6470
    3    0.6108             0.6537
    4    0.6213             0.6769
    5    0.5914             0.6492
Mean Macro-F1: 0.6041361274968313
Mean Balanced Accuracy: 0.6584563253848762
Std Macro-F1: 0.012365971217352901


In [69]:
from sklearn.model_selection import cross_validate
from sklearn.base import clone

cv_undersampled_results = cross_validate(
    clone(rf_undersampled),
    train_df[feature_columns],
    train_df["attack_cat"],
    groups=feature_hash,
    cv=cv,
    scoring={
        "macro_f1": "f1_macro",
        "balanced_accuracy": "balanced_accuracy"
    },
    n_jobs=1,
    error_score="raise"
)

cv_undersampled_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Macro-F1": cv_undersampled_results["test_macro_f1"],
    "Balanced Accuracy": cv_undersampled_results["test_balanced_accuracy"]
})

print(cv_undersampled_summary.round(4).to_string(index=False))
print("Mean Macro-F1:", cv_undersampled_summary["Macro-F1"].mean())
print("Mean Balanced Accuracy:", cv_undersampled_summary["Balanced Accuracy"].mean())
print("Std Macro-F1:", cv_undersampled_summary["Macro-F1"].std())

 Fold  Macro-F1  Balanced Accuracy
    1    0.6060             0.6117
    2    0.6191             0.6164
    3    0.5870             0.5928
    4    0.5934             0.5986
    5    0.5778             0.5902
Mean Macro-F1: 0.5966572965460764
Mean Balanced Accuracy: 0.6019387995684817
Std Macro-F1: 0.01619944588897537


In [70]:
from sklearn.model_selection import cross_validate
from sklearn.base import clone

cv_oversampled_results = cross_validate(
    clone(rf_oversampled),
    train_df[feature_columns],
    train_df["attack_cat"],
    groups=feature_hash,
    cv=cv,
    scoring={
        "macro_f1": "f1_macro",
        "balanced_accuracy": "balanced_accuracy"
    },
    n_jobs=1,
    error_score="raise"
)

cv_oversampled_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Macro-F1": cv_oversampled_results["test_macro_f1"],
    "Balanced Accuracy": cv_oversampled_results["test_balanced_accuracy"]
})

print(cv_oversampled_summary.round(4).to_string(index=False))
print("Mean Macro-F1:", cv_oversampled_summary["Macro-F1"].mean())
print("Mean Balanced Accuracy:", cv_oversampled_summary["Balanced Accuracy"].mean())
print("Std Macro-F1:", cv_oversampled_summary["Macro-F1"].std())

 Fold  Macro-F1  Balanced Accuracy
    1    0.5949             0.6287
    2    0.5958             0.6221
    3    0.5832             0.6094
    4    0.5938             0.6204
    5    0.5738             0.6047
Mean Macro-F1: 0.5883048638615895
Mean Balanced Accuracy: 0.6170460262703241
Std Macro-F1: 0.009563783356107927


In [71]:
from sklearn.model_selection import cross_validate
from sklearn.base import clone

cv_smotenc_results = cross_validate(
    clone(rf_smotenc),
    train_df[feature_columns],
    train_df["attack_cat"],
    groups=feature_hash,
    cv=cv,
    scoring={
        "macro_f1": "f1_macro",
        "balanced_accuracy": "balanced_accuracy"
    },
    n_jobs=1,
    error_score="raise"
)

cv_smotenc_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Macro-F1": cv_smotenc_results["test_macro_f1"],
    "Balanced Accuracy": cv_smotenc_results["test_balanced_accuracy"]
})

print(cv_smotenc_summary.round(4).to_string(index=False))
print("Mean Macro-F1:", cv_smotenc_summary["Macro-F1"].mean())
print("Mean Balanced Accuracy:", cv_smotenc_summary["Balanced Accuracy"].mean())
print("Std Macro-F1:", cv_smotenc_summary["Macro-F1"].std())

 Fold  Macro-F1  Balanced Accuracy
    1    0.5638             0.6021
    2    0.5759             0.6014
    3    0.5811             0.6044
    4    0.5637             0.5922
    5    0.5623             0.5956
Mean Macro-F1: 0.5693743581091427
Mean Balanced Accuracy: 0.599133187548962
Std Macro-F1: 0.008553794807385403


In [72]:
cv_models = {
    "Baseline": cv_control_summary,
    "Class Weight": cv_weighted_summary,
    "Random Oversampling": cv_oversampled_summary,
    "Random Undersampling": cv_undersampled_summary,
    "SMOTENC": cv_smotenc_summary
}

cv_comparison = pd.DataFrame([
    {
        "Method": name,
        "Mean Macro-F1": df["Macro-F1"].mean(),
        "Std Macro-F1": df["Macro-F1"].std(),
        "Mean Balanced Accuracy": df["Balanced Accuracy"].mean()
    }
    for name, df in cv_models.items()
])

output_dir = Path("../results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

cv_comparison.to_csv(
    output_dir / "cv_method_comparison.csv",
    index=False
)

print(cv_comparison.round(4).to_string(index=False))
print("Cross-validation comparison saved successfully!")

              Method  Mean Macro-F1  Std Macro-F1  Mean Balanced Accuracy
            Baseline         0.5616        0.0130                  0.5416
        Class Weight         0.6041        0.0124                  0.6585
 Random Oversampling         0.5883        0.0096                  0.6170
Random Undersampling         0.5967        0.0162                  0.6019
             SMOTENC         0.5694        0.0086                  0.5991
Cross-validation comparison saved successfully!
